Preprocess the Train Data


In [1]:
import pandas as pd
import numpy as np
import re
import spacy
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')

nlp = spacy.load("en_core_web_sm", disable=["ner", "parser"])

[nltk_data] Downloading package stopwords to C:\Users\akanksha
[nltk_data]     meshram\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [2]:
train_data = pd.read_csv('../data/raw/Restaurants_Train_v2.csv')

In [3]:
#cleaning the text data
def clean_text(text):
    #coverting to lower case
    text = text.lower()
    #removing special characters,punctuation and white spaces
    text = text.strip()
    cleaned = re.sub(r'[^a-zA-Z\s]', '', text)
    return cleaned


In [4]:
train_data['cleaned_sentence'] = train_data['Sentence'].apply(clean_text)

In [5]:
def preprocess_pipe(texts): #preprocessing the text data using nlp pipeline
    processed = []
    for doc in nlp.pipe(texts, batch_size=50):
        tokens = [token.lemma_ for token in doc 
                  if not token.is_stop 
                  and not token.is_punct 
                  and token.text.strip() != '']
        processed.append(' '.join(tokens))
    return processed

In [6]:
train_data['processed_sentence'] = preprocess_pipe(train_data['cleaned_sentence']) #calling the preprocess_pipe function to preprocess the cleaned sentences

In [7]:
aspect_map = {
    'food': [
        # existing +
        'food', 'pizza', 'pasta', 'sushi', 'burger', 'chicken', 'dish',
        'meal', 'menu', 'bread', 'sauce', 'salad', 'dessert', 'lunch',
        'dinner', 'breakfast', 'taste', 'flavour', 'flavor', 'paneer',
        'meat', 'fish', 'seafood', 'soup', 'sandwich', 'steak', 'rice',
        'noodle', 'appetizer', 'entree', 'portion', 'ingredient', 'spice',
        'fresh', 'cooked', 'fried', 'grilled', 'baked', 'roasted', 'raw',
        'sweet', 'salty', 'spicy', 'delicious', 'tasty', 'bland', 'cuisine',
        'bagel', 'bagels', 'cheese', 'dim sum', 'specials', 'quality',
        'wine', 'wine list', 'beer', 'drinks', 'drink', 'bar',
        # NEW
        'brunch', 'crust', 'rolls', 'roll', 'beef', 'dumplings', 'tuna',
        'toppings', 'sides', 'main course', 'quantity', 'sake', 'water',
        'dining', 'food quality', 'apps', 'appetizers', 'lamb', 'shrimp',
        'scallop', 'lobster', 'crab', 'turkey', 'pork', 'bacon', 'egg',
        'eggs', 'omelet', 'waffle', 'pancake', 'fries', 'chips', 'wrap',
        'gyoza', 'ramen', 'udon', 'tempura', 'maki', 'sashimi', 'nigiri',
        'hummus', 'falafel', 'shawarma', 'curry', 'naan', 'dosa', 'biryani'
    ],
    'service': [
        # existing +
        'service', 'staff', 'waiter', 'waitress', 'server', 'manager',
        'host', 'bartender', 'crew', 'attention', 'friendly', 'rude',
        'slow', 'fast', 'quick', 'attentive', 'helpful', 'professional',
        'courteous', 'efficient', 'prompt', 'responsive', 'kind', 'polite',
        'incompetent', 'negligent', 'welcoming', 'reception', 'hostess',
        'wait', 'reservation', 'reservations', 'owner', 'chef',
        'served', 'people', 'delivery',
        # NEW
        'check', 'seated', 'seat', 'management', 'employees',
        'seating', 'busboy', 'cashier', 'maitre', 'concierge',
        'table', 'tables', 'booking', 'experience', 'hospitality',
        'coffee', 'antipasti', 'pig feet', 'espresso', 'latte', 'cappuccino',
    ],
    'price': [
       'price', 'prices', 'cost', 'value', 'values', 'expensive', 'cheap',
       'worth', 'bill', 'bills', 'charge', 'money', 'affordable', 'overpriced',
       'reasonable', 'budget', 'pricey', 'costly', 'inexpensive', 'deal',
       'tip', 'tips', 'discount', 'rate', 'fee', 'payment', 'dollar', 
       'cash', 'tab', 'value for money', 'values for your money',
       'value ofr money', 'all you can eat', 'establishment'
    ],
    'ambience': [
        'ambience', 'ambiance', 'atmosphere', 'decor', 'music', 'noise',
        'environment', 'vibe', 'setting', 'seating', 'view', 'interior',
        'design', 'lighting', 'decoration', 'cozy', 'romantic', 'loud',
        'quiet', 'crowded', 'busy', 'clean', 'dirty', 'comfortable',
        'aesthetic', 'theme', 'layout', 'space', 'outdoor', 'indoor',
        # NEW
        'room', 'vibe', 'vibe', 'feel', 'look', 'decor', 'wall',
        'ceiling', 'floor', 'table', 'chair', 'window', 'garden',
        'patio', 'terrace', 'rooftop', 'basement', 'upstairs'
    ],
    'location': [
        'location', 'place', 'area', 'neighbourhood', 'neighborhood',
        'spot', 'venue', 'street', 'corner', 'parking', 'accessible',
        'convenient', 'downtown', 'nearby', 'distance', 'address',
        'situated', 'located', 'proximity', 'central', 'remote'
    ]
}

def map_aspect(aspect_term):
    term = str(aspect_term).lower().strip()
    for category, keywords in aspect_map.items():
        for keyword in keywords:
            if keyword.lower() in term:  # check if keyword is anywhere in term
                return category
    return 'others'

train_data['aspect_category'] = train_data['Aspect Term'].apply(map_aspect)
print(train_data['aspect_category'].value_counts())

aspect_category
food        2086
service      695
others       557
ambience     219
location      87
price         49
Name: count, dtype: int64


In [8]:
# Verify price terms are still being caught correctly
price_check = train_data[train_data['aspect_category'] == 'price']
print(f"\nPrice count: {len(price_check)}")
print(price_check['Aspect Term'].value_counts().head(15))


Price count: 49
Aspect Term
bill                     14
money                     6
tip                       6
value                     6
cost                      6
establishment             2
values for your money     1
value ofr money           1
bills                     1
tips                      1
all you can eat deal      1
feel                      1
Bill                      1
zen feel                  1
discount                  1
Name: count, dtype: int64


In [9]:
others = train_data[train_data['aspect_category'] == 'others']
print(f"Others count: {len(others)}")
print("\nTop unmatched aspect terms:")
print(others['Aspect Term'].value_counts().head(20))

Others count: 557

Top unmatched aspect terms:
Aspect Term
stuff        4
eats         4
salmon       4
kitchen      3
plate        3
pickles      3
mussels      3
courses      3
Pad Thai     3
counter      3
calamari     3
garlic       3
tiramisu     3
hot dogs     3
seasoning    3
shows        3
thai         3
caviar       3
serving      3
oil          3
Name: count, dtype: int64


In [10]:
train_data.to_csv('../data/processed/train_processed.csv', index=False)

1. we wrote a clean_text function and process_pipe function
2. Applied clean function to sentences and applied to processed pipeline to cleaned sentences
3. created a aspect_map with top 5 categories
4. applied map-aspect function to Aspect _term and created another category aspect category which shows values in top 5 aspeccts and rest in others
5. In the aspect category , foodd and other is top aspects
6. There are total 20 unmatched aspect categories
